In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#%pip install -r requirements.txt

In [4]:
%pip uninstall skimage scikit-image -y
%pip install scikit-image

Found existing installation: scikit-image 0.25.2
Uninstalling scikit-image-0.25.2:
  Successfully uninstalled scikit-image-0.25.2
Note: you may need to restart the kernel to use updated packages.
  Using cached scikit_image-0.25.2-cp310-cp310-macosx_10_9_x86_64.whl.metadata (14 kB)
Using cached scikit_image-0.25.2-cp310-cp310-macosx_10_9_x86_64.whl (14.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
print(np.__version__)

1.26.4


# Phase 1: Train & Validation Data Splitting
We need to randomly reserve 10% of the training images as our validation set.

In [3]:
import os
import glob
import shutil
import random

# Set seeds for reproducibility
random.seed(42)

In [4]:


def split_dataset(data_dir="data/train", val_dir="data/val", split_ratio=0.1):
    
    
    if not os.path.exists(data_dir):
        print(f"Source directory {data_dir} does not exist!")
        return
        
    os.makedirs(val_dir, exist_ok=True)
    
    # Get all uniquely named basenames (ignoring extensions)
    all_images = glob.glob(os.path.join(data_dir, "*.png"))
    base_names = [os.path.splitext(os.path.basename(img))[0] for img in all_images]
    
    
        
    # Calculate 10%
    num_val = int(len(base_names) * split_ratio)
    val_basenames = random.sample(base_names, num_val)
    
    print(f"Moving {num_val} records to {val_dir}...")
    
    # Move the groups (jpg, json, hea, dat)
    extensions = [".png", ".json", ".hea", ".dat"]
    count = 0
    for bn in val_basenames:
        for ext in extensions:
            src = os.path.join(data_dir, bn + ext)
            tgt = os.path.join(val_dir, bn + ext)
            if os.path.exists(src):
                shutil.move(src, tgt)
                count += 1
                
    print(f"Successfully moved {count} files to validation set.")

# Run the split
split_dataset()

Moving 270 records to data/val...
Successfully moved 1080 files to validation set.


# Phase 2: Dataloader Initialization
Now we initialize PyTorch DataLoaders pointing to `data/train` and `data/val`.

In [7]:
import torch
from torch.utils.data import DataLoader
from modeling.dataset import ECGDigitizationDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")

train_dataset = ECGDigitizationDataset(data_dir="data/train", split="train")
val_dataset = ECGDigitizationDataset(data_dir="data/val", split="val")

print(f"Training Records: {len(train_dataset)}")
print(f"Validation Records: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)


Using device: cpu
Training Records: 2430
Validation Records: 570


# Phase 3: Train Sub-Models for 2 Epochs
We use the existing `train.py` functions to instantiate models and loop over epochs.

In [8]:
from train import train_dotter_unet, train_resnet_unet

print("\n==============================")
print("Starting Stage 1: Dotter U-Net")
print("==============================")
dotter_model = train_dotter_unet(train_loader, val_loader, device=device, num_epochs=2)


print("\n==================================")
print("Starting Stage 2: ResNet34 U-Net")
print("==================================")
resnet_model = train_resnet_unet(train_loader, val_loader, device=device, num_epochs=2)


Starting Stage 1: Dotter U-Net

--- Starting Training: Dotter U-Net (Stage 1) ---


KeyboardInterrupt: 